In [ ]:
import math

def friction_factor_haaland(Re, roughness, diameter):
    """
    Returns the friction factor f using Haaland's Equation:
      1 / sqrt(f) = -1.8 * log10( ( (k/D)/3.7 )^1.11 + 6.9/Re )
    where k is the pipe roughness height.

    For laminar flow (Re < ~2000), we switch to 64/Re as an approximation.
    """
    if Re < 1e-6:
        # Guard against a nearly zero Reynolds number
        return 0.0
    if Re < 2000:
        # Laminar regime
        return 64.0 / Re

    term = ((roughness/diameter)/3.7)**1.11 + (6.9/Re)
    return 1.0 / ( -1.8 * math.log10(term) )**2


def main_equation_residual(v_b, param):
    """
    Computes the residual of the main pressure/energy equation:

      residual(v_b) = (del_p/(rho*g)) - [ del_h
                                          + v_b^2/(2*g)
                                          + 0.5*( ... f_A-based terms ... )*v_b^2
                                          + 0.5*( ... f_B-based terms ... )*v_b^2 ]

    where f_A, f_B are computed from a direct friction correlation.
    """
    # Unpack parameters for readability
    dp   = param['del_p']
    rho  = param['rho']
    g    = param['g']
    delh = param['del_h']
    dA   = param['d_A']
    D = param['D']
    LA   = param['L_A']
    nu   = param['visc']         # kinematic viscosity
    epsA = param['eps_A']        # roughness pipe B

    # Reynolds numbers for each pipe
    ReA = (v_b * dA) / nu

    # Compute friction factors via Haaland (or any direct correlation)
    fA = friction_factor_haaland(ReA, epsA, dA)

    # Right-hand side of your main equation
    # (matches the eq_main structure you provided)
    rhs = (delh
            + (v_b**2)/(2*g)
            + ((1 - (dA**2/D**2))**2
                + 6*0.08 + 2*1.1
                + fA*LA/dA
            )*(v_b**2)/(2*g)
          )

    lhs = dp/(rho*g)   # left-hand side

    return lhs - rhs   # residual = LHS - RHS


def solve_velocity_bisection(param, v_b_low=1e-5, v_b_high=7, tol=1e-6, max_iter=100):
    """
    Solve for v_b by bracket-bisection:
      1) Evaluate residual at v_b_low and v_b_high.
      2) Keep halving the bracket until residual is close to zero
         or we reach max_iter.

    Constraints:
      - main_equation_residual(v_b_low) and main_equation_residual(v_b_high)
        must have opposite signs (one positive, one negative).
    """
    f_low = main_equation_residual(v_b_low, param)
    f_high = main_equation_residual(v_b_high, param)

    if f_low * f_high > 0:
        raise ValueError(
            "Bisection method: The signs of residual at v_b_low and v_b_high must be different.\n"
            f"f_low={f_low}, f_high={f_high}"
        )

    for iteration in range(max_iter):
        v_b_mid = 0.5*(v_b_low + v_b_high)
        f_mid = main_equation_residual(v_b_mid, param)

        if abs(f_mid) < tol:
            # Converged
            return v_b_mid

        # Narrow down bracket based on the sign of the residual
        if (f_low * f_mid) < 0:
            # Root is between v_b_low and v_b_mid
            v_b_high = v_b_mid
            f_high = f_mid
        else:
            # Root is between v_b_mid and v_b_high
            v_b_low = v_b_mid
            f_low = f_mid

    # If we exit the loop, no convergence within max_iter
    raise ValueError("Bisection did not converge within max_iter iterations.")

param_values = {
    'del_p' : 250000,  # Pa
    'rho'   : 1000.0,         # kg/m^3
    'g'     : 0.001,            # m/s^2
    'del_h' : 100e-3,  # m
    'd_A'   : 6.53e-3,
    'D' : 12.7e-3,       # m
    'L_A'   : 100e-3,        # m
    'visc'  : 1.0e-6,         # m^2/s (kinematic viscosity = mu/rho)
    'eps_A' : 1e-6,         # Pipe roughness (m) for A
}
# 1) Choose a bracket for velocity in [0.00001, 5.0] m/s
# 2) Solve using bisection
v_b_solution = solve_velocity_bisection(param_values, v_b_low=1, v_b_high=70)

# Print the final velocity
print(f"Final velocity solution = {v_b_solution:.6f} m/s")
print(f"flow rate: {param_values['rho']*v_b_solution*math.pi*(param_values['d_A']/2)**2}")
print((v_b_solution * param_values['d_A']) / param_values['visc'])
#accumulator nominal volume: 0.25L
#water nominal volume: 0.180L => gas initial volume at 3.5bar = 0.007L

Final velocity solution = 10.511490 m/s
flow rate: 0.352030692326482
68640.03038753048
